# Transformer Assignment — Part (2)
## Self-Attention Scores in the Encoder

**Sentence:** *"What are the symptoms of diabetes?"*

**Student:** Abdelrahman Khalaf Ahmed Ghoneim  
**ID:** 4221407  
**Delta University for Science and Technology**


## Step 1: Import Libraries

In [1]:
import numpy as np

np.random.seed(42)
print("Libraries imported successfully.")


Libraries imported successfully.


## Step 2: Tokenization

Split the input sentence into tokens and define model dimensions.


In [2]:
sentence = "What are the symptoms of diabetes?"
tokens = sentence.replace("?", " ?").split()

seq_len = len(tokens)   # 7 tokens
d_model  = 8            # embedding dimension
d_k      = 4            # Q/K/V dimension per head

print("Sentence :", sentence)
print("Tokens   :", tokens)
print(f"seq_len={seq_len}  |  d_model={d_model}  |  d_k={d_k}")


Sentence : What are the symptoms of diabetes?
Tokens   : ['What', 'are', 'the', 'symptoms', 'of', 'diabetes', '?']
seq_len=7  |  d_model=8  |  d_k=4


## Step 3: Input Embedding + Positional Encoding




In [3]:
def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i]     = np.sin(pos / (10000 ** (i / d_model)))
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))
    return PE

# Random embeddings (simulates a lookup table)
X_embed = np.random.randn(seq_len, d_model)

# Add positional encoding
PE = positional_encoding(seq_len, d_model)
X  = X_embed + PE          # shape: (7, 8)

print("Input Embedding + Positional Encoding  X  (shape:", X.shape, ")")
print()
for i, tok in enumerate(tokens):
    print(f"  {tok:<12}  {np.round(X[i], 3)}")


Input Embedding + Positional Encoding  X  (shape: (7, 8) )

  What          [ 0.497  0.862  0.648  2.523 -0.234  0.766  1.579  1.767]
  are           [ 0.372  1.083 -0.364  0.529  0.252 -0.913 -1.724  0.438]
  the           [-0.104 -0.102 -0.709 -0.432  1.486  0.774  0.07  -0.425]
  symptoms      [-0.403 -0.879 -0.855  1.331 -0.571  0.708 -0.599  2.852]
  of            [-0.77  -1.711  1.212 -0.3    0.249 -0.96  -1.324  1.197]
  diabetes      [-0.22   0.455  0.364  0.576 -1.429  0.279 -0.456  2.057]
  ?             [ 0.064 -0.803  0.889  0.44  -0.617  1.61   1.037  1.931]


## Step 4: Learnable Weight Matrices W_Q, W_K, W_V

Three weight matrices are randomly initialized (in practice, these are learned during training).


In [4]:
W_Q = np.random.randn(d_model, d_k)   # (8, 4)
W_K = np.random.randn(d_model, d_k)   # (8, 4)
W_V = np.random.randn(d_model, d_k)   # (8, 4)

print("W_Q shape:", W_Q.shape)
print("W_K shape:", W_K.shape)
print("W_V shape:", W_V.shape)


W_Q shape: (8, 4)
W_K shape: (8, 4)
W_V shape: (8, 4)


## Step 5: Compute Q, K, V Matrices



In [5]:
Q = X @ W_Q    # (7, 4)
K = X @ W_K    # (7, 4)
V = X @ W_V    # (7, 4)

print("Q shape:", Q.shape)
print()
print("Q Matrix:\n", np.round(Q, 3))
print()
print("K Matrix:\n", np.round(K, 3))
print()
print("V Matrix:\n", np.round(V, 3))


Q shape: (7, 4)

Q Matrix:
 [[-1.092 -1.981  4.712  2.032]
 [-0.999 -1.319 -3.748  2.576]
 [-0.255  1.699 -4.134 -1.849]
 [-1.547 -4.48   4.668  1.219]
 [ 1.574  1.996 -0.159  5.244]
 [-1.017 -3.408  4.583 -0.324]
 [-0.415 -0.995  6.069 -1.017]]

K Matrix:
 [[-3.979  0.753 -1.346 -1.212]
 [-2.209 -5.07   2.488 -1.686]
 [ 0.507  1.187  0.993  0.502]
 [-1.167 -5.794  0.785  1.875]
 [ 1.984 -3.881  4.64   2.284]
 [-0.707 -4.534 -1.187 -0.275]
 [ 0.679  0.271 -2.07   2.29 ]]

V Matrix:
 [[-0.027  5.141 -6.413  1.727]
 [-0.293 -0.025  3.442  1.61 ]
 [-0.944  0.485  1.225 -2.44 ]
 [-0.393  0.99   0.261  5.679]
 [-1.307 -5.816  2.695  3.347]
 [ 1.654  2.366 -2.386  4.094]
 [ 1.327  1.631 -4.786  1.464]]


## Step 6: Scaled Dot-Product Attention



In [6]:
def softmax(x):
    """Numerically stable row-wise softmax."""
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# Raw attention scores
scores = Q @ K.T / np.sqrt(d_k)     # (7, 7)

# Attention weights after softmax
attn_weights = softmax(scores)       # (7, 7)

# Final attended output
output = attn_weights @ V            # (7, 4)

print("Scores shape       :", scores.shape)
print("Attention weights  :", attn_weights.shape)
print("Output shape       :", output.shape)


Scores shape       : (7, 7)
Attention weights  : (7, 7)
Output shape       : (7, 4)


## Step 7: Display Raw Attention Scores

In [7]:
print("Raw Attention Scores  Q·K^T / sqrt(d_k)  (7 × 7):")
print()
header = f"{'':12}" + "".join(f"{t:>11}" for t in tokens)
print(header)
for i, tok in enumerate(tokens):
    row = "".join(f"{scores[i,j]:11.3f}" for j in range(seq_len))
    print(f"  {tok:<12}{row}")


Raw Attention Scores  Q·K^T / sqrt(d_k)  (7 × 7):

                   What        are        the   symptoms         of   diabetes          ?
  What             -2.977     10.375      1.397     10.131     16.013      1.803     -3.190
  are               2.452     -2.389     -2.249      5.346     -4.186      5.213      6.312
  the               5.049     -7.608     -1.572     -8.128    -15.251     -1.055      2.306
  symptoms         -2.491     17.844     -0.429     16.857     19.380      7.768     -4.569
  of               -5.450    -11.417      2.822     -1.848      3.308     -5.708      6.974
  diabetes         -2.150     15.736     -0.087     11.963     15.868      5.412     -5.922
  ?                -3.018     11.387      2.061      4.553     14.437     -1.059     -7.723


## Step 8: Display Attention Weights (after Softmax)

In [8]:
print("Attention Weights after Softmax  (7 × 7):")
print("Each row sums to 1.0")
print()
print(header)
for i, tok in enumerate(tokens):
    row = "".join(f"{attn_weights[i,j]:11.3f}" for j in range(seq_len))
    print(f"  {tok:<12}{row}")

# Verify rows sum to 1
print()
print("Row sums:", np.round(attn_weights.sum(axis=1), 4))


Attention Weights after Softmax  (7 × 7):
Each row sums to 1.0

                   What        are        the   symptoms         of   diabetes          ?
  What              0.000      0.004      0.000      0.003      0.994      0.000      0.000
  are               0.012      0.000      0.000      0.219      0.000      0.192      0.576
  the               0.936      0.000      0.001      0.000      0.000      0.002      0.060
  symptoms          0.000      0.166      0.000      0.062      0.772      0.000      0.000
  of                0.000      0.000      0.015      0.000      0.025      0.000      0.960
  diabetes          0.000      0.462      0.000      0.011      0.527      0.000      0.000
  ?                 0.000      0.045      0.000      0.000      0.955      0.000      0.000

Row sums: [1. 1. 1. 1. 1. 1. 1.]


## Step 9: Self-Attention Output

In [9]:
print("Self-Attention Output  (7 × 4):")
print()
for i, tok in enumerate(tokens):
    print(f"  {tok:<12}  {np.round(output[i], 3)}")


Self-Attention Output  (7 × 4):

  What          [-1.301 -5.776  2.691  3.347]
  are           [ 0.996  1.674 -3.237  2.896]
  the           [ 0.057  4.918 -6.297  1.711]
  symptoms      [-1.082 -4.433  2.668  3.203]
  of            [ 1.228  1.431 -4.511  1.452]
  diabetes      [-0.829 -3.068  3.014  2.569]
  ?             [-1.261 -5.554  2.729  3.269]


## Step 10: Interpretation




In [10]:
# Find the token each word attends to most
print("Most attended token per query token:")
print()
for i, tok in enumerate(tokens):
    top_j = np.argmax(attn_weights[i])
    print(f"  '{tok}' attends most to '{tokens[top_j]}'  "
          f"(weight = {attn_weights[i, top_j]:.3f})")


Most attended token per query token:

  'What' attends most to 'of'  (weight = 0.994)
  'are' attends most to '?'  (weight = 0.576)
  'the' attends most to 'What'  (weight = 0.936)
  'symptoms' attends most to 'of'  (weight = 0.772)
  'of' attends most to '?'  (weight = 0.960)
  'diabetes' attends most to 'of'  (weight = 0.527)
  '?' attends most to 'of'  (weight = 0.955)
